<a href="https://colab.research.google.com/github/Innovatewithapple/CNNProjects/blob/main/RealFakeHumanFaces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input, Dropout

In [2]:
import os
from google.colab import userdata
import cv2
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image

In [28]:
from sklearn.metrics import classification_report
from tensorflow.keras.optimizers import Adam

In [21]:
from google.colab import files

In [3]:
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

print("Kaggle API activated via Secrets!")

Kaggle API activated via Secrets!


In [5]:
# Paste the copied command here, starting with an exclamation mark
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:27<00:00, 146MB/s]



In [6]:
!unzip -q 140k-real-and-fake-faces.zip -d ./dataset_folder

In [7]:
train_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/train'
test_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/test'
validation_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/valid'

In [8]:
train_gen = ImageDataGenerator(rescale= 1./255)
test_gen = ImageDataGenerator(rescale= 1./255)
validation_gen = ImageDataGenerator(rescale= 1./255)

In [9]:
train_data = train_gen.flow_from_directory(train_dir,
                                           target_size=(128,128),
                                           batch_size= 32,
                                           class_mode='binary')

test_data = train_gen.flow_from_directory(test_dir,
                                           target_size=(128,128),
                                           batch_size= 32,
                                           class_mode='binary')

validation_data = train_gen.flow_from_directory(validation_dir,
                                           target_size=(128,128),
                                           batch_size= 32,
                                           class_mode='binary')

Found 100000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [10]:
x_batch, y_batch = next(train_data)
y_batch.shape

(32,)

In [11]:
num_classes = len(np.unique(y_batch))
num_classes

2

In [12]:
print(train_data.class_indices)

{'fake': 0, 'real': 1}


In [13]:
# 1. Define the "Safety Net"
early_stop = EarlyStopping(
    monitor='val_loss',   # Watch the validation loss
    patience=10,           # How many epochs to wait for improvement before quitting
    restore_best_weights=True # Keep the version of the model that performed best
)

**64 Filters**

In [16]:
model = Sequential()
model.add(Input(shape=(128,128,3)))
model.add(Conv2D(32,kernel_size=(3,3),strides=(1,1),activation='relu'))
model.add(Conv2D(32,kernel_size=(3,3),strides=(1,1),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=(2,2)))

model.add(Conv2D(64,kernel_size=(3,3),strides=(1,1),activation='relu'))
model.add(Conv2D(64,kernel_size=(3,3),strides=(1,1),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=(2,2)))

model.add(Flatten())
model.add(Dense(128,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
# model.fit(
#     train_data,
#     validation_data=validation_data,
#     steps_per_epoch=20,     # Limit training to 20 batches
#     validation_steps=5,     # Limit validation to 5 batches
#     epochs=1)               # Just do one epoch)
model.fit(train_data,validation_data=validation_data,epochs=10,callbacks=[early_stop])

Epoch 1/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 174s 55ms/step - accuracy: 0.7961 - loss: 0.4198 - val_accuracy: 0.8975 - val_loss: 0.2497
Epoch 2/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 169s 54ms/step - accuracy: 0.9329 - loss: 0.1687 - val_accuracy: 0.9419 - val_loss: 0.1514
Epoch 3/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 161s 52ms/step - accuracy: 0.9687 - loss: 0.0844 - val_accuracy: 0.9516 - val_loss: 0.1370
Epoch 4/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 159s 51ms/step - accuracy: 0.9827 - loss: 0.0473 - val_accuracy: 0.9541 - val_loss: 0.1324
Epoch 5/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 160s 51ms/step - accuracy: 0.9877 - loss: 0.0346 - val_accuracy: 0.9549 - val_loss: 0.1382
Epoch 6/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 157s 50ms/step - accuracy: 0.9901 - loss: 0.0272 - val_accuracy: 0.9597 - val_loss: 0.1588
Epoch 7/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 154s 49ms/step - accuracy: 0.9914 - loss: 0.0239 - val_accuracy: 0.9615 - val_loss: 0.1308
Epoch 8/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 157s 50ms/step - accuracy: 

In [18]:
# 1. Force the generator to NOT shuffle
test_data.shuffle = False

# 2. Reset it so it starts at the very first image
test_data.reset()

# 3. NOW predict
y_pred_probs = model.predict(test_data)
y_true = test_data.classes # This will now match the order of y_pred_probs
y_pred = (y_pred_probs > 0.5).astype("int32")

625/625 ━━━━━━━━━━━━━━━━━━━━ 34s 54ms/step


In [19]:
print('Classification Report: ', classification_report(test_data.classes, y_pred, target_names=['Fake', 'Real']))

Classification Report:                precision    recall  f1-score   support

        Fake       0.97      0.96      0.96     10000
        Real       0.96      0.97      0.96     10000

    accuracy                           0.96     20000
   macro avg       0.96      0.96      0.96     20000
weighted avg       0.96      0.96      0.96     20000



In [20]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 124, 124, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 60, 60, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 58, 58, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 29, 29, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 53824)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     6,889,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,865,893 (79.60 MB)

 Trainable params: 6,955,297 (26.53 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 13,910,596 (53.06 MB)

In [22]:
# Saves the model to Colab's temporary local storage
model.save('cnnfakereal64filter.h5')

In [23]:
files.download('cnnfakereal64filter.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**128 Filters**

In [25]:
model_v2 = Sequential()
model_v2.add(Input(shape=(128,128,3)))
model_v2.add(Conv2D(32,kernel_size=(3,3),strides=(1,1),activation='relu'))
model_v2.add(Conv2D(32,kernel_size=(3,3),strides=(1,1),activation='relu'))
model_v2.add(MaxPooling2D(pool_size=(2,2),strides=(2,2)))

model_v2.add(Conv2D(64,kernel_size=(3,3),strides=(1,1),activation='relu'))
model_v2.add(Conv2D(64,kernel_size=(3,3),strides=(1,1),activation='relu'))
model_v2.add(MaxPooling2D(pool_size=(2,2),strides=(2,2)))

model_v2.add(Conv2D(128,kernel_size=(3,3),strides=(1,1),activation='relu'))
model_v2.add(Conv2D(128,kernel_size=(3,3),strides=(1,1),activation='relu'))
model_v2.add(MaxPooling2D(pool_size=(2,2),strides=(2,2)))

model_v2.add(Flatten())
model_v2.add(Dense(256,activation='relu'))
model_v2.add(Dropout(0.5))
model_v2.add(Dense(1,activation='sigmoid'))

model_v2.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

model_v2.fit(train_data,validation_data=validation_data,epochs=10,callbacks=[early_stop])

Epoch 1/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 174s 54ms/step - accuracy: 0.4993 - loss: 0.6933 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 2/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 162s 52ms/step - accuracy: 0.4989 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6931
Epoch 3/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 164s 52ms/step - accuracy: 0.4993 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 4/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 167s 53ms/step - accuracy: 0.5024 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 5/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 166s 53ms/step - accuracy: 0.4977 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6932


In [26]:
model_v2.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 124, 124, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 60, 60, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 58, 58, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 29, 29, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 27, 27, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 25, 25, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │     4,718,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,018,341 (57.29 MB)

 Trainable params: 5,006,113 (19.10 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 10,012,228 (38.19 MB)

Version 3

In [27]:
model_v3 = Sequential()
model_v3.add(Input(shape=(128,128,3)))

# Block 1 - Keeps size at 128x128
model_v3.add(Conv2D(32, (3,3), activation='relu', padding='same'))
model_v3.add(Conv2D(32, (3,3), activation='relu', padding='same'))
model_v3.add(MaxPooling2D((2,2))) # Shinks to 64x64

# Block 2 - Keeps size at 64x64
model_v3.add(Conv2D(64, (3,3), activation='relu', padding='same'))
model_v3.add(Conv2D(64, (3,3), activation='relu', padding='same'))
model_v3.add(MaxPooling2D((2,2))) # Shrinks to 32x32

# Block 3 - Keeps size at 32x32
model_v3.add(Conv2D(128, (3,3), activation='relu', padding='same'))
model_v3.add(Conv2D(128, (3,3), activation='relu', padding='same'))
model_v3.add(MaxPooling2D((2,2))) # Shrinks to 16x16

model_v3.add(Flatten()) # Now 16*16*128 = 32,768 neurons
model_v3.add(Dense(256, activation='relu'))
# model_v3.add(Dropout(0.5)) # Keep this commented until we see movement
model_v3.add(Dense(1, activation='sigmoid'))


model_v3.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

model_v3.fit(train_data,validation_data=validation_data,epochs=10,callbacks=[early_stop])

Epoch 1/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 180s 54ms/step - accuracy: 0.7832 - loss: 0.4390 - val_accuracy: 0.8822 - val_loss: 0.2827
Epoch 2/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 172s 55ms/step - accuracy: 0.9064 - loss: 0.2269 - val_accuracy: 0.9132 - val_loss: 0.2158
Epoch 3/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 167s 53ms/step - accuracy: 0.9419 - loss: 0.1480 - val_accuracy: 0.9288 - val_loss: 0.1765
Epoch 4/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 169s 54ms/step - accuracy: 0.9602 - loss: 0.1030 - val_accuracy: 0.9370 - val_loss: 0.1652
Epoch 5/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 171s 55ms/step - accuracy: 0.9715 - loss: 0.0744 - val_accuracy: 0.9394 - val_loss: 0.1831


In [29]:
model_v4 = Sequential()
model_v4.add(Input(shape=(128,128,3)))

# Block 1 - Keeps size at 128x128
model_v4.add(Conv2D(32, (3,3), activation='relu', padding='same'))
model_v4.add(Conv2D(32, (3,3), activation='relu', padding='same'))
model_v4.add(MaxPooling2D((2,2))) # Shinks to 64x64

# Block 2 - Keeps size at 64x64
model_v4.add(Conv2D(64, (3,3), activation='relu', padding='same'))
model_v4.add(Conv2D(64, (3,3), activation='relu', padding='same'))
model_v4.add(MaxPooling2D((2,2))) # Shrinks to 32x32

# Block 3 - Keeps size at 32x32
model_v4.add(Conv2D(128, (3,3), activation='relu', padding='same'))
model_v4.add(Conv2D(128, (3,3), activation='relu', padding='same'))
model_v4.add(MaxPooling2D((2,2))) # Shrinks to 16x16

model_v4.add(Flatten()) # Now 16*16*128 = 32,768 neurons
model_v4.add(Dense(256, activation='relu'))
model_v3.add(Dropout(0.5)) # Keep this commented until we see movement
model_v4.add(Dense(1, activation='sigmoid'))

my_optimizer = Adam(learning_rate=0.0001)

# You hand that "engine" to the model here
model_v4.compile(optimizer=my_optimizer,
              loss='binary_crossentropy',
              metrics=['accuracy'])

#model_v4.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

model_v4.fit(train_data,validation_data=validation_data,epochs=10,callbacks=[early_stop])

Epoch 1/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 177s 55ms/step - accuracy: 0.7792 - loss: 0.4520 - val_accuracy: 0.8692 - val_loss: 0.3077
Epoch 2/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 191s 61ms/step - accuracy: 0.9070 - loss: 0.2290 - val_accuracy: 0.9279 - val_loss: 0.1783
Epoch 3/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 172s 55ms/step - accuracy: 0.9498 - loss: 0.1307 - val_accuracy: 0.9475 - val_loss: 0.1326
Epoch 4/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 173s 55ms/step - accuracy: 0.9698 - loss: 0.0801 - val_accuracy: 0.9576 - val_loss: 0.1091
Epoch 5/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 173s 55ms/step - accuracy: 0.9813 - loss: 0.0497 - val_accuracy: 0.9625 - val_loss: 0.1039
Epoch 6/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 175s 56ms/step - accuracy: 0.9879 - loss: 0.0330 - val_accuracy: 0.9632 - val_loss: 0.1040
Epoch 7/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 172s 55ms/step - accuracy: 0.9905 - loss: 0.0256 - val_accuracy: 0.9658 - val_loss: 0.0997
Epoch 8/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 175s 56ms/step - accuracy: 

In [30]:
model_v4.save('cnnfakereal128filter.h5')

In [31]:
files.download('cnnfakereal128filter.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
model_v4.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_20 (Conv2D)              │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_22 (Conv2D)              │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_24 (Conv2D)              │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_25 (Conv2D)              │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │     8,388,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,028,389 (99.29 MB)

 Trainable params: 8,676,129 (33.10 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 17,352,260 (66.19 MB)